In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, mesh_energy, param_utils, viewer
import numpy as np
import sim_utils

import matplotlib
from matplotlib import pyplot as plt
import visualization

import newton_flow

In [ ]:
m_rest = mesh.Mesh('data/pants_rest.obj')
m_defo = mesh.Mesh('data/pants_deformed.obj')

In [ ]:
v = mesh_energy.NodalVars(m_rest, 2)
v.setVars(m_rest.vertices().ravel())

In [ ]:
em = MeshFEM.EmbeddedMesh(m_rest, v)

In [ ]:
view = viewer.Viewer(em, wireframe=True)
view.setCameraParams(((1.0,-0.5, 7.5), (0.0, 1.0, 0.0), (1.0, -0.5, 0.0)))
view.show()

In [ ]:
v.setVars(m_defo.vertices().ravel())

In [ ]:
view.update()

In [ ]:
nf = newton_flow.symmetric_dirichlet(m_rest, v)
# nf = newton_flow.linear_elastic(m_rest, v)

In [ ]:
import py_newton_optimizer
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [nf])

In [ ]:
# Nullspace pinning strategy
# TODO: add epsilon * low rank.
FIX_VARS = True
if FIX_VARS:
    fv = sim_utils.getBBoxVars(m_rest, sim_utils.BBoxFace.MIN_X)
    prob.setFixedVars(fv)
else:
    # prob.hessianShift = 1e-5
    nf.elementHessianShift = 1e-6

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
# opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()
# opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
def ground_truth_flow(step_size, x_init = None, verbose=False):
    flow_vertices = []
    if x_init is None: x_init = m_defo.vertices().ravel()
    flow_vertices.append(x_init)
    prob.setVars(x_init)
    while np.linalg.norm(prob.gradient()) > 1e-6:
        d = opt.newton_step()
        flow_vertices.append(prob.getVars())
        curr_energy = prob.energy()
        alpha = step_size
        x = prob.getVars()
        while True:
            prob.setVars(x + alpha * d)
            if (prob.energy() > curr_energy):
                alpha = 0.5 * alpha
            else: break
        if verbose: print(len(flow_vertices) - 1, np.linalg.norm(d), prob.hessianWasProjected, alpha)
    prob.setVars(x_init)
    return np.array([fv.reshape(-1, 2) for fv in flow_vertices])

In [ ]:
x_start = v.getVars()

In [ ]:
import visualization
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
# fv = ground_truth_flow(1.0, x_start)
# print(len(fv))

fv = ground_truth_flow(0.02, x_start, verbose=True)
print(len(fv))

In [ ]:
def eval_trajectory(x_0, coeffs, alphas):
    result = []
    for a in alphas:
        x = x_0.copy()
        for i in range(len(coeffs)):
            x += coeffs[i] * a**(i + 1)
        result.append(x.reshape(-1, 2))
    return np.array(result)

In [ ]:
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

def flow_frame(frame):
    visualization.plot_mesh(fv[0].reshape(-1,2), m_rest.elements(), zorder=-1, face_color='white')
    visualization.plot_mesh(fv[frame].reshape(-1,2), m_rest.elements(), ax=plt.gca())
    visualization.plot_trajectory(fv, color='gray', alpha=0.5)
    # visualization.plot_vector_field(fv[frame], ds[frame].reshape(-1, 2), ax=plt.gca(), quiver_scale=1)
    
    # Also plot each Taylor extrapolation up to the specified degree.
    max_degree = 4
    prob.setVars(fv[frame].ravel())
    d = opt.newton_step()
    proj = prob.hessianWasProjected
    prob.setVars(fv[frame].ravel())
    d_coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, max_degree, proj)
    # print(d_coeffs)
    for deg in range(1, len(d_coeffs) + 1):
        alphas = np.linspace(0, 3.5, 100)
        visualization.plot_trajectory(eval_trajectory(fv[frame].ravel(), d_coeffs[:deg], alphas), color=colors[deg - 1])
    
    plt.text(0.01, 0.01, f"Step {frame} (⍺={0.02 * frame:0.3})", transform=plt.gca().transAxes, ha="left", va="bottom")
    
    plt.xlim(-1, 2.25)
    plt.ylim(-1.84, 1.5)

In [ ]:
!rm videos/newton_flow/*.png
for frame in range(0, len(fv)):
    flow_frame(frame)
    plt.savefig(f'videos/newton_flow/frame_{frame}.png', dpi=150, facecolor='white', transparent=False)
    plt.close()

In [ ]:
!ffmpeg \
    -f image2 \
    -framerate 24 \
    -i videos/newton_flow/frame_%d.png \
    -c:v libx264 \
    -preset veryslow \
    -crf 10 \
    -pix_fmt yuv420p \
    -y videos/newton_flow_alphamax_1.5_project_on_demand.mp4

In [ ]:
prob.setVars(m_defo.vertices().ravel())
opt.options.niter = 3
opt.optimize()
d = opt.newton_step()
view.update(vectorField=d.reshape(m_rest.numNodes(), -1))

In [ ]:
x_start = prob.getVars()

In [ ]:
fv, ds = ground_truth_flow(0.02, x_start)

In [ ]:
prob.setVars(x_start)

In [ ]:
opt.update_factorizations()
d_coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, False)

In [ ]:
# Simple automated method:
# Try search over degree (up to 4) at alpha = 1; backtrack if necessary.
# Test verson with projected Hessian.

In [ ]:
def eval_trajectory(x_0, coeffs, alphas):
    result = []
    for a in alphas:
        x = x_0.copy()
        for i in range(len(coeffs)):
            x += coeffs[i] * a**(i + 1)
        result.append(x.reshape(-1, 2))
    return np.array(result)

In [ ]:
alphas = np.linspace(0, 1.0, 100)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for deg in range(1, 12):
    visualization.plot_mesh(x_start.reshape(-1,2), m_rest.elements())
    trajectory = eval_trajectory(x_start, d_coeffs[:deg], alphas)
    visualization.plot_trajectories(trajectory, color=colors[(deg - 1) % 10])
    visualization.plot_trajectories(fv, color='gray', zorder=0, lw=3)
    plt.xlim(-1.25, 2.25)
    plt.ylim(-1.84, 1.5)
    plt.savefig(f'trajectory_deg_{deg}.png', dpi=100, transparent=False)
    plt.close()

In [ ]:
!open .

In [ ]:
# Energy line search visualization
emin = np.inf
for deg in range(1, 12):
    trajectory = eval_trajectory(x_start, d_coeffs[:deg], alphas)
    energies = []
    gnorms = []
    for uv in trajectory:
        prob.setVars(uv.ravel())
        energies.append(prob.energy())
        gnorms.append(np.linalg.norm(prob.gradient()))
    emin = min(emin, min(energies))
    plt.plot(alphas, energies, label=f'Deg {deg}')
plt.ylim(energies[0] - (energies[0] - emin) * 1.05, energies[0] + (energies[0] - emin) * 1.05)
plt.legend(loc='center left', bbox_to_anchor=[1.0, 0.5])
plt.xlabel('Line Search Parameter ⍺')
plt.ylabel('Energy')
plt.tight_layout()
plt.savefig('line_search_energy.pdf')

# TODO
- arc length version
- Postprocess Newton step to remove rigid motion (does this make the steps more coherent?)
- Try KKT formulation for pinning rigid motion
- See relative performance of symmetric indefinite factorization within Accelerate
- Consider sparse + epsilon * low rank factorization idea for pinning rigid motion (omitting epsilon^2 fully dense term)

# Finite Difference Validation

In [ ]:
import benchmark
prob.disableCaching = True
benchmark.reset()
x = prob.getVars()
eps = 0.001
prob.setVars(x + eps * d)
d_plus = opt.newton_step()
d_coeffs_plus = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, False)

prob.setVars(x - eps * d)
d_minus = opt.newton_step()
d_coeffs_minus = nf.computeTaylorCoefficients(opt.hessian_factorization, 16, False)
d_prime_ad = (d_plus - d_minus) / (2 * eps)
# Note: this finite difference approximation is wrong! we must incorporate the effect of `d_prime_ad` when differencing!
d_pprime_ad = (d_plus + d_minus - 2 * d) / (eps * eps) 

prob.setVars(x + eps * d + 0.5 * (eps * eps) * d_prime_ad)
d_pp = opt.newton_step()
prob.setVars(x - eps * d + 0.5 * (eps * eps) * d_prime_ad)
d_mm = opt.newton_step()
d_pprime_ad2 = (d_pp + d_mm - 2 * d) / (eps * eps)
prob.setVars(x)
# benchmark.report()

In [ ]:
import math
i = 6
((d_coeffs_plus[i][:5] - d_coeffs_minus[i][:5]) / (2 * eps)) / (d_coeffs[i + 1][:5] * math.factorial(i + 2) / math.factorial(i + 1))